# ميزان — واجهة Gradio الاحترافية

شغّل الخلايا بالترتيب. النموذج المدرّب مرفق؛ لا تحتاج إعادة تدريب أو GPU أو حساب ngrok.
تُنشأ مشاركة Gradio مؤقتة عند إتاحتها، وليست استضافة دائمة. الرابط عام لمن يملكه، ويتوقف بانتهاء الجلسة أو انتهاء صلاحية خدمة Gradio.
إذا تعذر إنشاؤه، يبقى عرض الواجهة عبر وكيل Colab متاحًا دون رابط عام.
الدرجات غير معايرة، وعتبة الحقيقي 55%؛ نتيجة اختبار المشروع 88.45% ولا تثبت أصالة المعدن.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, time, hashlib, urllib.request, urllib.error

REPOSITORY_URL = "https://github.com/Aishaas1/AI-porject-new.git"
REPO = Path("/content/AI-porject-new")
RUNTIME = Path("/content/gold-runtime")
RUN_DIR = Path("/content/gold-run")
MODEL_NAME = "gold_selected.pt"
PORT = 7860
CREATE_PUBLIC_LINK = True

print("إصدار Python في Colab:", sys.version.split()[0])
print("الاستدلال سيعمل على CPU؛ لا يلزم تفعيل GPU.")


## 2. تنزيل المشروع
تُستخدم النسخة الموجودة إذا نُفذت الخلية سابقًا؛ لا تُحذف تعديلاتك ولا يحدث تحديث تلقائي.
لتحديث نسخة سابقة يدويًا: نفّذ `git pull origin main` داخل المجلد بعد حفظ تعديلاتك.


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO)], check=True)
else:
    if not (REPO / ".git").exists():
        raise RuntimeError("المجلد موجود لكنه ليس المستودع المتوقع. اختر مسارًا جديدًا في الإعدادات.")
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != REPOSITORY_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("المجلد يخص مستودعًا مختلفًا؛ لن أستبدله.")
    print("استخدام النسخة الموجودة؛ لم تُستبدل ملفاتها.")

PROJECT_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("نسخة المشروع:", PROJECT_COMMIT)
assert (REPO / "app.py").is_file(), "ملف التطبيق غير موجود"
RUN_DIR.mkdir(parents=True, exist_ok=True)


## 3. تثبيت المكتبات في بيئة معزولة
لا نستبدل PyTorch أو NumPy في جلسة Colab الرئيسية؛ يستخدم التطبيق مفسرًا مستقلًا.
إذا كان Python الرئيسي خارج 3.10–3.12، تُستخدم أداة uv لإنشاء بيئة Python 3.12.
قد يستغرق التثبيت الأول عدة دقائق. لا تُعطّل التحقق من TLS أو بصمات التنزيل عند حدوث خطأ.


In [ ]:
if not (RUNTIME / "bin" / "python").exists():
    if (3, 10) <= sys.version_info[:2] <= (3, 12):
        subprocess.run([sys.executable, "-m", "venv", str(RUNTIME)], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
        subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", "--python", "3.12", str(RUNTIME)], check=True)

RUNTIME_PY = str(RUNTIME / "bin" / "python")
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV.update(KERAS_BACKEND="torch", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
subprocess.run([RUNTIME_PY, "-m", "pip", "install", "torch==2.5.1", "torchvision==0.20.1",
                "--index-url", "https://download.pytorch.org/whl/cpu"], check=True, env=RUNTIME_ENV)
subprocess.run([RUNTIME_PY, "-m", "pip", "install", "-r", str(REPO / "requirements-gradio.txt")], check=True, env=RUNTIME_ENV)
subprocess.run([RUNTIME_PY, "-m", "pip", "check"], check=True, env=RUNTIME_ENV)
subprocess.run([RUNTIME_PY, "-c", "import torch, torchvision, flask, keras, gradio; print('Runtime ready:', torch.__version__, torchvision.__version__)"],
               check=True, env=RUNTIME_ENV)
print("البيئة جاهزة؛ لا يلزم إعادة تشغيل جلسة Colab الرئيسية.")


## 4. التحقق من الأوزان
تُقارن بصمة النموذج بما هو محفوظ مع المشروع، ثم تُختبر بنية النموذج دون تنزيل ImageNet مجددًا.
أي اختلاف يوقف التشغيل بدل استعمال ملف غير متوافق.


In [ ]:
MODEL_PATH = REPO / "artifacts" / MODEL_NAME
if not MODEL_PATH.is_file():
    raise FileNotFoundError("الأوزان المرفقة غير موجودة. تأكد من تنزيل آخر نسخة من المستودع.")
selection = json.loads((REPO / "reports" / "candidate_selection.json").read_text())
actual_hash = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
if actual_hash != selection["checkpoint_sha256"]:
    raise RuntimeError("بصمة الأوزان تختلف عن تقرير الاختيار؛ افحص النسخة قبل تشغيلها.")

check_model = """
import json, sys
from model import load_model
model = load_model(sys.argv[1])
print(json.dumps({'architecture': model.architecture, 'preprocessing': model.preprocessing,
                  'decision_threshold': model.decision_threshold}))
"""
verified = subprocess.check_output([RUNTIME_PY, "-c", check_model, str(MODEL_PATH)],
                                   cwd=REPO, env=RUNTIME_ENV, text=True)
print("بصمة النموذج:", actual_hash)
print("إعدادات النموذج:", verified.strip())


## تشغيل Gradio والحصول على رابط المشاركة
اترك `CREATE_PUBLIC_LINK=True` للحصول على رابط مؤقت من Gradio إن سمحت الشبكة.
لا يلزم رمز. السجل محفوظ خارج المستودع. إعادة هذه الخلية توقف العملية السابقة لهذا الدفتر فقط.
قد يتأخر رابط المشاركة عن جاهزية الخادم المحلي؛ إذا لم يظهر راجع سجل النفق في الخلية التالية أو استخدم العرض الداخلي.


In [ ]:
import socket, re
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=15)
if "server_log" in globals() and not server_log.closed:
    server_log.close()
with socket.socket() as probe:
    probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        probe.bind(("127.0.0.1", PORT))
    except OSError:
        raise RuntimeError("المنفذ مشغول؛ غيّر PORT أو أوقف الخادم الذي تملكه.") from None
LOG_PATH = RUN_DIR / "gradio.log"
server_log = LOG_PATH.open("w")
SERVER_ENV = RUNTIME_ENV.copy()
SERVER_ENV.update(GOLD_MODEL_PATH=str(MODEL_PATH), GRADIO_ANALYTICS_ENABLED="False")
command = [RUNTIME_PY, "-u", "gradio_app.py", "--port", str(PORT)]
if CREATE_PUBLIC_LINK:
    command.append("--share")
server = subprocess.Popen(command, cwd=REPO, env=SERVER_ENV, stdout=server_log, stderr=subprocess.STDOUT)
BASE_URL = f"http://127.0.0.1:{PORT}"
ready = False
for _ in range(120):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen(BASE_URL + "/config", timeout=3) as response:
            config = json.load(response)
        if config.get("title") == "ميزان | قراءة ذكية للذهب":
            ready = True
            break
    except (urllib.error.URLError, OSError, TimeoutError):
        pass
    time.sleep(1)
if not ready:
    print(LOG_PATH.read_text(errors="replace")[-10000:])
    if server.poll() is None:
        server.terminate()
        server.wait(timeout=15)
    server_log.close()
    raise RuntimeError("لم يبدأ Gradio؛ راجع السجل أعلاه.")
print("واجهة Gradio جاهزة محليًا.")
public_url = None
if CREATE_PUBLIC_LINK:
    for _ in range(45):
        log_text = LOG_PATH.read_text(errors="replace")
        urls = re.findall(r"https://[a-zA-Z0-9.-]+\.gradio\.live", log_text)
        if urls:
            public_url = urls[-1]
            break
        if "Could not create share link" in log_text:
            break
        time.sleep(1)
if public_url:
    print("افتح واجهة Gradio:", public_url)
else:
    print("لم يتوفر رابط عام. استخدم عرض Colab الداخلي في الخلية التالية؛ الخادم ما زال يعمل.")


## عرض الواجهة داخل Colab — دون خدمة مشاركة
يمكن استخدام هذا العرض حتى عند فشل رابط Gradio. يعتمد ظهور الإطار على المتصفح وجلسة Colab.
إذا فشل الإطار أيضًا، ما زال ملف Gold_Classifier_Colab.ipynb الآخر يحتوي واجهة رفع صور داخل الدفتر.


In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=1000)
except Exception:
    print("عرض Colab غير متاح في هذه الواجهة. استخدم رابط المشاركة إن ظهر أو دفتر واجهة رفع الصور المباشرة.")


## السجل وإيقاف الخدمة
لا توقف هذه الخلية البرنامج عند «تشغيل الكل». فعّل STOP_APP عند الانتهاء فقط.
لا يُضمن أن خدمة المشاركة متاحة في كل منطقة أو شبكة. لتشغيل دائم أنشئ Hugging Face Space بمشروعك؛ يحتاج ذلك حسابك وصلاحية النشر.


In [ ]:
SHOW_LOG = False
STOP_APP = False
if SHOW_LOG:
    print(LOG_PATH.read_text(errors="replace")[-10000:])
if STOP_APP:
    if server.poll() is None:
        server.terminate()
        server.wait(timeout=15)
    server_log.close()
    print("أُوقفت واجهة Gradio ورابطها.")
else:
    print("الواجهة مستمرة ما دامت جلسة Colab تعمل.")
